### 1. Read and prepare the cleaned dataset ###

In [1]:
### Import necessary libraries ###
import time
import pandas as pd
import numpy as np
import tqdm.auto as tqdm
import os

import helper_func as hf

from functools import lru_cache

import pubchempy as pcp

from rdkit import Chem
from descriptastorus.descriptors import rdNormalizedDescriptors

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_path = './Cleaned_VLE_Data.csv'
df = pd.read_csv(data_path)
print(df.head())

                      property    value phase  Temperature, K  Pressure, kPa  \
0  Thermal conductivity, W/m/K  0.02096   Gas           300.0          724.0   
1  Thermal conductivity, W/m/K  0.02125   Gas           300.0         1288.0   
2  Thermal conductivity, W/m/K  0.02161   Gas           300.0         1835.0   
3  Thermal conductivity, W/m/K  0.02197   Gas           300.0         2335.0   
4  Thermal conductivity, W/m/K  0.02245   Gas           300.0         2820.0   

   Mole fraction     Component 1 Component 2  
0         0.2493  carbon dioxide     methane  
1         0.2493  carbon dioxide     methane  
2         0.2493  carbon dioxide     methane  
3         0.2493  carbon dioxide     methane  
4         0.2493  carbon dioxide     methane  


In [3]:
df_oh, cat_map = hf.one_hot_encode(df.drop(columns=['Component 1', 'Component 2']), drop_first=True, dummy_na=False, prefix_sep="_")
df_oh['Component 1'] = df['Component 1']
df_oh['Component 2'] = df['Component 2']

In [4]:
@lru_cache(maxsize=1_000_000)
def get_smiles(name, retries=3):
    for i in range(retries):
        try:
            compounds = pcp.get_compounds(name, 'name')
            if compounds:
                return compounds[0].isomeric_smiles
            return None
        except Exception as e:
            if "SSL" in str(e) or "EOF" in str(e):
                print(f"SSL Error for {name}, retrying ({i+1}/{retries})...")
                time.sleep(2) # Give the connection a moment to breathe
                continue
            print(f"Permanent Error fetching {name}: {e}")
            break
    return None

# Apply the cached function to get SMILES for both components
df_oh['Smiles 1'] = df_oh['Component 1'].apply(get_smiles)
df_oh['Smiles 2'] = df_oh['Component 2'].apply(get_smiles)

# Replace empty strings with NaN across the specific columns
df_oh['Smiles 1'] = df_oh['Smiles 1'].replace('', np.nan)
df_oh['Smiles 2'] = df_oh['Smiles 2'].replace('', np.nan)

# Check for empty/NaN values in 'Smiles 1' and 'Smiles 2'
empty_count = (df_oh['Smiles 1'].isnull() | df_oh['Smiles 2'].isnull()).sum()
print(f"Number of rows with empty/NaN 'Smiles 1' or 'Smiles 2': {empty_count}")

# Drop rows with empty/NaN values in 'Smiles 1' or 'Smiles 2'
if empty_count > 0:
    df_oh = df_oh.dropna(subset=['Smiles 1', 'Smiles 2'])
    print(f"New shape after dropping: {df_oh.shape}")

df_oh['mol1'] = df_oh['Smiles 1'].apply(Chem.MolFromSmiles)
df_oh['mol2'] = df_oh['Smiles 2'].apply(Chem.MolFromSmiles)

/var/folders/vt/h1pypxb11hjc5hwp_kcs2xy00000gn/T/ipykernel_1443/2311382139.py:7: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  return compounds[0].isomeric_smiles
/var/folders/vt/h1pypxb11hjc5hwp_kcs2xy00000gn/T/ipykernel_1443/2311382139.py:7: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  return compounds[0].isomeric_smiles


Number of rows with empty/NaN 'Smiles 1' or 'Smiles 2': 2241
New shape after dropping: (108142, 34)


[00:45:46] WARNING: not removing hydrogen atom without neighbors
[00:45:46] WARNING: not removing hydrogen atom without neighbors
[00:45:46] WARNING: not removing hydrogen atom without neighbors
[00:45:46] WARNING: not removing hydrogen atom without neighbors
[00:45:46] WARNING: not removing hydrogen atom without neighbors
[00:45:46] WARNING: not removing hydrogen atom without neighbors
[00:45:46] WARNING: not removing hydrogen atom without neighbors
[00:45:46] WARNING: not removing hydrogen atom without neighbors
[00:45:46] WARNING: not removing hydrogen atom without neighbors
[00:45:46] WARNING: not removing hydrogen atom without neighbors
[00:45:46] WARNING: not removing hydrogen atom without neighbors
[00:45:46] WARNING: not removing hydrogen atom without neighbors
[00:45:46] WARNING: not removing hydrogen atom without neighbors
[00:45:46] WARNING: not removing hydrogen atom without neighbors
[00:45:46] WARNING: not removing hydrogen atom without neighbors
[00:45:46] WARNING: not r

In [5]:
def get_descriptors(data_set:pd.DataFrame,
                    mol_col:list,
                    save_dir: str = "./processed_data" , #= './descriptors' Path to save the descriptor files
                    save_file: bool = True, # Whether to save the updated DataFrame with descriptors as a CSV file
                    output_df: bool = True, # Whether to return the updated DataFrame with descriptors
                    )->pd.DataFrame | None:
    """Helper function to calculate and save molecular descriptors for specified columns in a DataFrame."""

    @lru_cache(maxsize=10_000_000)
    def get_cached_descriptors(mol_input): return generator.calculateMol(mol_input, None)
    generator = rdNormalizedDescriptors.RDKit2DNormalized()

    if save_file: os.makedirs(save_dir, exist_ok=True)

    # Create a copy of the dataset to avoid modifying the original
    df = data_set.copy()
    for col in mol_col:
        print(f"Processing {col}...")
        x = np.stack([get_cached_descriptors(m) for m in tqdm.tqdm(df[col].tolist())])
        print(f"{col} shape: {x.shape}")
        if save_file: np.save(f'{save_dir}/descriptors_for_{col}.npy', x)
        df = pd.concat([df, pd.DataFrame(x, columns=[f"{col}_desc_{i}" for i in range(x.shape[1])])], axis=1)

    # Save the updated DataFrame with descriptors
    if save_file: df.to_csv(f'{save_dir}/dataset_with_descriptors.csv', index=False)
    return df if output_df else None

In [ ]:
df_processed = get_descriptors(df_oh, mol_col=['mol1', 'mol2'])

Processing mol1...


  1%|          | 1017/108142 [00:06<11:19, 157.64it/s]